In [1]:
import vitaldb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
CASE_ID = 1

# 1. Fetch the track list
df_trks = pd.read_csv('https://api.vitaldb.net/trks')
available_tracks = df_trks[df_trks['caseid'] == CASE_ID]['tname'].tolist()

print(f"Downloading data for Patient {CASE_ID}...")
vals = vitaldb.load_case(CASE_ID, available_tracks, interval=1)
df = pd.DataFrame(vals, columns=available_tracks)

# 2. CLEANING: Remove attributes with > 80% missing data
min_valid_count = int(len(df) * 0.20)
df = df.dropna(axis=1, thresh=min_valid_count)

valid_tracks = df.columns.tolist()
print(f"Reduced from {len(available_tracks)} to {len(valid_tracks)} tracks after removing sparse ones (>80% missing).\n")

# Helper function to guess units based on parameter names
def get_unit(track_name):
    track_upper = track_name.upper()
    if 'HR' in track_upper: return "(bpm)"
    if 'SPO2' in track_upper: return "(%)"
    if 'BP' in track_upper or 'ART' in track_upper or 'CVP' in track_upper: return "(mmHg)"
    if 'TEMP' in track_upper or 'BT' in track_upper: return "(°C)"
    if 'CO2' in track_upper: return "(mmHg)"
    if 'RR' in track_upper or 'RESP' in track_upper: return "(rpm)"
    if 'MV' in track_upper: return "(L/min)"
    if 'TV' in track_upper: return "(mL)"
    return "(Value)"

# 3. Plotting Setup
time_minutes = np.arange(len(df)) / 60.0
num_tracks = len(valid_tracks)

fig, axes = plt.subplots(num_tracks, 1, figsize=(12, 2.5 * num_tracks), sharex=True)

if num_tracks == 1:
    axes = [axes]

for i, track_name in enumerate(valid_tracks):
    ax = axes[i]
    y_data = df[track_name]

    # Plot the data
    ax.plot(time_minutes, y_data, linewidth=1, color='#1f77b4')

    # --- Y-AXIS OUTLIER HANDLING ---
    y_min = y_data.quantile(0.01)
    y_max = y_data.quantile(0.99)

    if pd.notna(y_min) and pd.notna(y_max) and (y_max > y_min):
        margin = (y_max - y_min) * 0.1  # Add a 10% visual margin
        ax.set_ylim(y_min - margin, y_max + margin)

    # --- X-AXIS ALIGNMENT ---
    ax.set_xlim(0, time_minutes[-1])

    # --- TITLES & LABELS ---
    param_name = track_name.split('/')[-1] if '/' in track_name else track_name
    unit = get_unit(param_name)

    ax.set_title(track_name, loc='left', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.4)
    ax.set_ylabel(f"{param_name}\n{unit}", fontsize=10)

    # Hide top/right borders
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.xlabel("Time (Minutes)", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()